In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-11-19 21:30:34,437 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-11-19 21:30:34,447 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [13]:
additional_articles = [
    {
        "title": "How Prorated Billing Works When Upgrading or Downgrading",
        "content": "When a user switches between Basic and Premium mid-cycle, CultPass charges or credits the prorated difference for the remaining days in the billing period.\n\n**Suggested phrasing:**\n\"When you change plans mid-cycle, your invoice is automatically prorated based on the remaining days in your current billing period.\"",
        "tags": "billing, prorata, upgrade, downgrade, subscription"
    },
    {
        "title": "Refund Eligibility and Policy",
        "content": "Subscription fees are non-refundable once a billing cycle has started, except in cases of verified duplicate charges, technical outages preventing access, or cancelled premium events.\n\n- Refunds under $50 can be processed automatically after verification.\n- Refunds over $50 require escalation to a billing specialist.\n\n**Suggested phrasing:**\n\"I can help check your refund eligibility. If an event was cancelled or you experienced a duplicate charge, we'll review and process it per our policy.\"",
        "tags": "refund, billing, policy, duplicate charge"
    },
    {
        "title": "How to Cancel an Event Reservation",
        "content": "Users can cancel a reservation in the CultPass app under 'My Reservations' up to 24 hours before the event start time without losing their monthly quota slot.\n\n- Cancellations within 24 hours forfeit the quota slot unless the event was cancelled by the venue.\n\n**Suggested phrasing:**\n\"You can cancel your reservation in 'My Reservations' at least 24 hours before the event to have your monthly quota credit restored.\"",
        "tags": "reservation, cancel, quota, events"
    },
    {
        "title": "Why Is My Account Blocked?",
        "content": "Accounts may be temporarily blocked due to repeated failed payment attempts, suspected unauthorized sharing, or policy violations.\n\n- Check user `is_blocked` status in CultPass.\n- Escalate to human Account Access team to verify identity before unblocking.\n\n**Suggested phrasing:**\n\"Your account currently has a security or billing hold. I am escalating this to our Account Access team so they can verify your details and restore access.\"",
        "tags": "blocked, access, security, account, escalation"
    },
    {
        "title": "Monthly Experience Quota and Rollover Rules",
        "content": "Basic plans include 4 experiences per month; Premium plans include up to 10. Unused experience slots do not roll over to the next billing cycle and reset on your renewal date.\n\n**Suggested phrasing:**\n\"Your experience quota resets on your monthly billing renewal date, and unused slots do not roll over.\"",
        "tags": "quota, subscription, rollover, limits"
    },
    {
        "title": "QR Code Not Scanning or Missing at Venue",
        "content": "If a user's QR code is not visible or fails to scan:\n- Ask them to refresh the 'My Reservations' tab and increase screen brightness.\n- Confirm they are logged into the same email used for the booking.\n- Provide their 6-character `reservation_id` so the venue can check them in manually.\n\n**Suggested phrasing:**\n\"Try refreshing 'My Reservations' and turning up your screen brightness. You can also give the venue staff your reservation ID for manual check-in.\"",
        "tags": "qr code, reservation, check-in, venue, troubleshooting"
    },
    {
        "title": "App Outage or Service Disruption Steps",
        "content": "When users report errors loading experiences, payments failing globally, or 500 errors, check the current service status feed.\n- Confirm whether an active incident is ongoing.\n- Share the latest status update and estimated recovery time.\n\n**Suggested phrasing:**\n\"We're currently investigating a service disruption affecting the CultPass app. Our engineering team is working on a fix, and we'll update you as soon as it's resolved.\"",
        "tags": "outage, incident, status, downtime, error"
    },
    {
        "title": "Updating Your Registered Email Address",
        "content": "Users can update their email in the CultPass app under 'My Account' > 'Profile Settings'. A verification link will be sent to the new address. If they no longer have access to their old email, escalate to Account Support.\n\n**Suggested phrasing:**\n\"You can change your email under 'My Account' > 'Profile Settings' and confirm via the verification link sent to your new address.\"",
        "tags": "email, account, profile, settings"
    },
    {
        "title": "Booking Premium Experiences and Extra Fees",
        "content": "Premium experiences (marked with a star badge) may require a supplemental fee or a Premium subscription tier. Any extra fee is displayed clearly before confirming the reservation.\n\n**Suggested phrasing:**\n\"Premium experiences are either included with a Premium tier or require a small supplemental fee, which is always shown before you confirm.\"",
        "tags": "premium, experiences, pricing, reservation"
    },
    {
        "title": "Group or Companion Bookings",
        "content": "CultPass memberships are individual, but Premium members can bring 1 guest to select experiences using an extra quota slot when available.\n\n**Suggested phrasing:**\n\"CultPass memberships are individual, though Premium members can use an extra quota slot to bring a guest on eligible experiences.\"",
        "tags": "group, guest, companion, premium, reservation"
    },
]

In [17]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

cultpass_articles.extend(additional_articles)

In [18]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [19]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [20]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [21]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [22]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [23]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [24]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [25]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='f169a96b-76f5-4b98-8708-329021a996f7', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='fd315282-e2a3-4acb-9fd3-e4a2b24fc0b4', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='4258b675-6b1a-49e4-b59b-380fc9acdacc', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='8702f6f1-dddb-48c4-9010-3acc6b61df08', title='How to Handle Login Issues?')>
<Knowledge(article_id='de42c734-c903-40f6-9d76-731c45b2d92b', title='How Prorated Billing Works When Upgrading or Downgrading')>
<Knowledge(article_id='0545efe8-b434-47d7-bd4a-188b847193eb', title='Refund Eligibility and Policy')>
<Knowledge(article_id='a8d414cd-d416-419d-988b-8823687b2be6', title='How to Cancel an Event Reservation')>
<Knowledge(article_id='eaab23a5-c356-4874-a8dd-c02de15e1c4d', title='Why Is My Account Blocked?')>
<Knowledge(article_id='7ca8f4c8-a8a3-482e-b3bd-96d3efac5806', title='Monthly Experience Quota and Rollover Rules')>
<Knowledge(arti

In [26]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='7a0e71cb-7c93-495a-a4d8-8ccb9865e33b', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [27]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='6a637c85-1f0f-4e63-9860-a7737e9a5031', role='user', content='I can't log in to my Cultpass ...')>
